# Modelo Causal — DAG (Directed Acyclic Graph)

**TCC:** Estimativa de Efeitos Causais na Jornada de Compra de um Marketplace  
**Objetivo do notebook:** Construir e formalizar o grafo causal (DAG) que representa as relações entre as variáveis operacionais da jornada do pedido e a avaliação do cliente.

---

## Contexto

O DAG é a etapa que **precede e fundamenta** todas as análises causais (IPTW, RDD).  
Ele responde à pergunta: *quais variáveis influenciam o tratamento e o outcome, e como elas se relacionam?*

| Elemento | Variável | Definição |
|---|---|---|
| **Treatment** | `is_delayed` | 1 se pedido entregue após data estimada, 0 caso contrário |
| **Outcome** | `review_score_outcome` | 1 = avaliação neutra/positiva (score 3-5); 0 = negativa (score 1-2) |
| **Confundidores** | ver `features.yaml` | Variáveis que afetam simultaneamente tratamento e outcome |

---

## Seções

1. Imports
2. Variáveis do modelo (via `features.yaml`)
3. Justificativa das arestas causais
4. DAG Geral
5. DAG por tratamento — critério de backdoor
6. Formalização via DoWhy
7. Resumo e análise

## 1. Imports

In [3]:
import os
import warnings
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from app.config.settings import INTERIM_DATA_DIR, PROJECT_DIR
from app.data import get_features
from app.data.utils import find_specific_variables

warnings.filterwarnings('ignore')
FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'

print('Imports OK')

ModuleNotFoundError: No module named 'pandas'

## 2. Variáveis do modelo

As variáveis são carregadas diretamente do `features.yaml`, garantindo consistência com todo o pipeline do projeto.

| Papel | Descrição |
|---|---|
| `treatment` | Variável de intervenção (`is_delayed`) |
| `outcome` | Variável de desfecho binarizada (`review_score_outcome`) |
| `confounder` | Variáveis que afetam simultaneamente tratamento e outcome |

In [ ]:
# Carrega features.yaml
features = get_features('features.yaml')

# Extrai variáveis por papel no modelo causal
treatment    = find_specific_variables(features, 'treatment',   specific_value=True)[0]
outcome      = find_specific_variables(features, 'outcome',     specific_value=True)[0]
confounders  = find_specific_variables(features, 'confounder',  specific_value=True)

print(f'Treatment  : {treatment}')
print(f'Outcome    : {outcome}')
print(f'Confounders ({len(confounders)}): {confounders}')

## 3. Justificativa das arestas causais

As arestas representam **mecanismos causais plausíveis** — não correlações estatísticas.
Cada aresta foi definida com base no contexto operacional de logística de e-commerce.

### 3.1 Confundidores → Tratamento (`is_delayed`)

| Grupo | Variável | Mecanismo causal |
|---|---|---|
| **Preço/Valor** | `total_price`, `avg_price` | Produtos de alto valor têm requisitos logísticos especiais (seguros, embalagens) → maior complexidade → mais atrasos |
| **Preço/Valor** | `total_payment`, `avg_payment`, `installment_value` | Pedidos de maior valor total envolvem mais etapas logísticas |
| **Frete** | `total_freight`, `avg_freight` | Frete baixo = serviço logístico mais lento; frete alto pode indicar produto pesado ou destino remoto |
| **Complexidade** | `n_items` | Mais itens → separação e conferência mais complexas → maior risco de atraso |
| **Complexidade** | `n_items_missing_info` | Dados faltantes indicam sellers problemáticos → maior risco de atraso operacional |
| **Complexidade** | `n_item_distinct_categ` | Múltiplas categorias = múltiplos sellers → coordenação mais difícil → atrasos |
| **Complexidade** | `n_payment` | Múltiplos meios de pagamento indicam pedidos maiores e mais complexos |
| **Dimensões** | `avg_weight` | Produtos pesados têm manuseio mais difícil e são mais sujeitos a atrasos |
| **Dimensões** | `avg_lenght`, `avg_height`, `avg_width` | Dimensões físicas determinam o tipo de transporte e a capacidade de alocação nos centros logísticos |
| **Temporal** | `purchase_weekday` | Compras no final de semana atrasam o início do processamento e despacho |
| **Temporal** | `purchase_month` | Meses de alta demanda (Natal, Black Friday) sobrecarregam a logística → mais atrasos |

### 3.2 Confundidores → Outcome (`review_score_outcome`)

| Grupo | Variável | Mecanismo causal |
|---|---|---|
| **Preço/Valor** | `avg_price`, `total_price` | Produtos mais caros geram expectativas maiores — qualquer problema é percebido como mais grave |
| **Preço/Valor** | `total_payment`, `avg_payment`, `installment_value` | O valor total investido amplifica a expectativa de qualidade e pontualidade |
| **Frete** | `avg_freight`, `total_freight` | Frete alto representa custo adicional que eleva a expectativa; frete inesperado gera insatisfação |
| **Complexidade** | `n_items` | Pedidos com mais itens têm maior probabilidade de problemas parciais (item faltando, embalagem ruim) |
| **Complexidade** | `n_item_distinct_categ` | Diversidade de categorias = experiências fragmentadas = mais pontos de insatisfação |
| **Complexidade** | `n_items_missing_info` | Informações faltantes no pedido podem gerar confusão na recepção do produto |
| **Complexidade** | `n_payment` | Pedidos mais complexos têm mais pontos de falha na experiência do cliente |
| **Dimensões** | `avg_weight`, `avg_lenght`, `avg_height`, `avg_width` | Produtos pesados/volumosos são mais difíceis de receber e manusear → afetam percepção da experiência |
| **Temporal** | `purchase_month` | Sazonalidade afeta o humor e as expectativas do cliente (ex: presentes de Natal) |
| **Temporal** | `purchase_weekday` | O contexto temporal da compra afeta a expectativa inicial do cliente |

### 3.3 Tratamento → Outcome

| Aresta | Mecanismo causal |
|---|---|
| `is_delayed` → `review_score_outcome` | Atraso na entrega frustra diretamente o cliente, sendo o principal driver de avaliação negativa |

## 4. DAG Geral

O DAG abaixo representa as relações causais entre as 17 variáveis confundidoras,
o tratamento `is_delayed` e o outcome `review_score_outcome`.

**Estrutura em 3 camadas:**
- **Topo:** 17 confundidores agrupados por categoria semântica (cor)
- **Meio:** tratamento `is_delayed`
- **Base:** outcome `review_score_outcome`

As arestas cinzas representam confundidores → T e confundidores → Y.  
A aresta vermelha representa o efeito causal de interesse: T → Y.

In [ ]:
# Grupos de confundidores para coloração no DAG
CONF_PRECO = ['total_price', 'avg_price', 'total_payment', 'avg_payment', 'installment_value']
CONF_FRETE = ['total_freight', 'avg_freight']
CONF_COMPL = ['n_items', 'n_items_missing_info', 'n_item_distinct_categ', 'n_payment']
CONF_DIMS  = ['avg_weight', 'avg_lenght', 'avg_height', 'avg_width']
CONF_TEMP  = ['purchase_weekday', 'purchase_month']

CONF_ORDER = CONF_PRECO + CONF_FRETE + CONF_COMPL + CONF_DIMS + CONF_TEMP  # ordem de exibição

# Todas as arestas do DAG (todos os 17 confundidores afetam T e Y)
ARESTAS_DAG = (
    [(c, treatment) for c in confounders] +
    [(c, outcome)   for c in confounders] +
    [(treatment, outcome)]
)

# Mapeamento de cores por grupo
GRUPO_CORES = {
    **{v: '#6a1b9a' for v in CONF_PRECO},   # roxo
    **{v: '#1565c0' for v in CONF_FRETE},   # azul
    **{v: '#e65100' for v in CONF_COMPL},   # laranja
    **{v: '#2e7d32' for v in CONF_DIMS},    # verde
    **{v: '#00838f' for v in CONF_TEMP},    # teal
}

def plot_dag_v12(arestas, conf_ordered, grupo_cores, treatment, outcome,
                 titulo='DAG Causal v12', figsize=(26, 9)):
    G = nx.DiGraph()
    G.add_edges_from(arestas)

    # Layout em 3 camadas
    pos = {}
    for i, c in enumerate(conf_ordered):
        pos[c] = (i * 1.85, 2)
    x_center = (len(conf_ordered) - 1) * 1.85 / 2
    pos[treatment] = (x_center, 1)
    pos[outcome]   = (x_center, 0)

    cor_map = {n: grupo_cores.get(n, '#455a64') for n in G.nodes()}
    cor_map[treatment] = '#b71c1c'
    cor_map[outcome]   = '#1b5e20'
    cores = [cor_map[n] for n in G.nodes()]

    edge_colors = ['#b71c1c' if u == treatment else '#90a4ae' for u, v in G.edges()]

    fig, ax = plt.subplots(figsize=figsize)
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color=cores, node_size=1800,
        font_size=5.5, font_color='white', font_weight='bold',
        edge_color=edge_colors, arrows=True,
        arrowsize=12, arrowstyle='->',
        connectionstyle='arc3,rad=0.08',
        width=1.1
    )

    patches = [
        mpatches.Patch(color='#6a1b9a', label='Preço/Valor'),
        mpatches.Patch(color='#1565c0', label='Frete'),
        mpatches.Patch(color='#e65100', label='Complexidade do pedido'),
        mpatches.Patch(color='#2e7d32', label='Dimensões físicas'),
        mpatches.Patch(color='#00838f', label='Temporal'),
        mpatches.Patch(color='#b71c1c', label='Tratamento (is_delayed)'),
        mpatches.Patch(color='#1b5e20', label='Outcome (review_score_outcome)'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=8, ncol=2)
    ax.set_title(titulo, fontsize=13, pad=15)
    ax.axis('off')
    plt.tight_layout()
    return fig

print(f'Total de arestas no DAG : {len(ARESTAS_DAG)}')
print(f'  conf → tratamento     : {len(confounders)}')
print(f'  conf → outcome        : {len(confounders)}')
print(f'  tratamento → outcome  : 1')

fig = plot_dag_v12(
    ARESTAS_DAG, CONF_ORDER, GRUPO_CORES, treatment, outcome,
    titulo='DAG v12 — is_delayed → review_score_outcome  |  17 confundidores',
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / 'dag_v12_geral.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 5. Critério de backdoor

O **critério de backdoor** garante que, ao condicionar nos confundidores,
todos os caminhos de backdoor entre `is_delayed` e `review_score_outcome` são bloqueados.

Um conjunto S satisfaz o critério de backdoor em relação a (T, Y) se:
1. **Nenhum nó em S é descendente de T**
2. **S bloqueia todos os caminhos de backdoor** entre T e Y (caminhos que entram em T por uma aresta "de trás")

> **Neste DAG:** todos os 17 confundidores são causas *comuns* de T e Y.  
> Nenhum deles é descendente de `is_delayed` — portanto o critério é satisfeito pelo conjunto completo.

In [ ]:
# Verifica critério de backdoor formalmente via NetworkX
G_back = nx.DiGraph()
G_back.add_edges_from(
    [(c, treatment) for c in confounders] +
    [(c, outcome)   for c in confounders] +
    [(treatment, outcome)]
)

print('Propriedades do DAG:')
print(f'  Nós              : {G_back.number_of_nodes()}')
print(f'  Arestas          : {G_back.number_of_edges()}')
print(f'  É acíclico (DAG) : {nx.is_directed_acyclic_graph(G_back)}')

descendants_T = sorted(nx.descendants(G_back, treatment))
ancestors_T   = sorted(nx.ancestors(G_back, treatment))

print(f'\nAntecessores de "{treatment}": {ancestors_T}')
print(f'Descendentes de  "{treatment}": {descendants_T}')

any_conf_is_descendant = any(c in descendants_T for c in confounders)
print(f'\nAlgum confundidor é descendente de "{treatment}"? {any_conf_is_descendant}')
print(f'\n→ Critério de backdoor {"SATISFEITO" if not any_conf_is_descendant else "VIOLADO"}:')
print(  '  Todos os confundidores são causas comuns de T e Y,')
print(  '  nenhum é descendente do tratamento.')

## 6. Formalização via DoWhy

O **DoWhy** verifica formalmente se o efeito causal `is_delayed → review_score_outcome`
é identificável dado o conjunto de ajuste (os 17 confundidores).

O estimando esperado é o **ATE via backdoor adjustment**:

$$\text{ATE} = E[Y(1) - Y(0)] = \sum_c \bigl(E[Y \mid T{=}1, C{=}c] - E[Y \mid T{=}0, C{=}c]\bigr) \cdot P(C{=}c)$$

In [ ]:
import dowhy
from dowhy import CausalModel

# Carrega dataset
df_raw = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, 'interim_dataset.parquet'))
df_raw = df_raw[df_raw['order_status'] == 'delivered'].copy()
print(f'Dataset bruto: {df_raw.shape[0]:,} pedidos delivered')

# Verifica disponibilidade das variáveis
cols_needed    = confounders + [treatment, outcome]
cols_missing   = [c for c in cols_needed if c not in df_raw.columns]
available_conf = [c for c in confounders if c in df_raw.columns]

print(f'Confundidores disponíveis: {len(available_conf)} / {len(confounders)}')
if cols_missing:
    print(f'Colunas ausentes: {cols_missing}')

# Dataset para o modelo
cols_model = available_conf + [treatment, outcome]
df_model   = df_raw[cols_model].dropna().copy()
print(f'Dataset modelagem (sem NaN): {df_model.shape[0]:,} linhas')

# Constrói o grafo GML para o DoWhy
def build_gml(treatment, outcome, confounders):
    nos     = set(confounders + [treatment, outcome])
    arestas = ([(c, treatment) for c in confounders] +
               [(c, outcome)   for c in confounders] +
               [(treatment, outcome)])
    lines = ['graph [directed 1']
    for no in nos:
        lines.append(f'  node [ id "{no}" label "{no}" ]')
    for u, v in arestas:
        lines.append(f'  edge [ source "{u}" target "{v}" ]')
    lines.append(']')
    return '\n'.join(lines)

gml   = build_gml(treatment, outcome, available_conf)
model = CausalModel(data=df_model, treatment=treatment,
                    outcome=outcome, graph=gml)
print('\nModelo DoWhy criado com sucesso.')

In [ ]:
# Executa DoWhy: identifica o efeito causal
identified = model.identify_effect(proceed_when_unidentifiable=True)
print('\n' + '=' * 60)
print(f'DoWhy — {treatment} → {outcome}')
print('=' * 60)
print(identified)

## 7. Resumo e análise

### 7.1 Interpretação do DAG

O DAG é a **fundamentação teórica** que justifica por que a análise IPTW pode ser interpretada
como causal — e não apenas correlacional.

Ele **declara explicitamente as suposições causais** que não podem ser verificadas nos dados:

- *"Acredito que `avg_price` afeta tanto o atraso quanto a avaliação"*
- *"Acredito que não existe variável oculta U que afete simultaneamente `is_delayed` e `review_score_outcome`"* (suposição de não-confundimento)

### 7.2 Estrutura do DAG v12

| Elemento | Detalhe |
|---|---|
| **Tratamento** | `is_delayed` — 1 se entregue após a data estimada |
| **Outcome** | `review_score_outcome` — 1 = avaliação neutra/positiva, 0 = negativa |
| **Confundidores** | 17 variáveis em 5 grupos semânticos |
| **Arestas** | 17 conf→T + 17 conf→Y + 1 T→Y = **35 arestas** |

### 7.3 Critério de backdoor — satisfeito

| Condição | Status |
|---|---|
| Nenhum confundidor é descendente de `is_delayed` | Satisfeito |
| O conjunto bloqueia todos os caminhos de backdoor | Satisfeito |
| DoWhy confirma identificabilidade via backdoor | Confirmado |

### 7.4 Próximos notebooks

| Notebook | Objetivo |
|---|---|
| `results_01_consolidacao.ipynb` | Tabela de ATEs, Love Plot, dose-resposta |
| `results_02_simulacao_contrafactual.ipynb` | Simulação contrafactual |
| `results_03_robustez.ipynb` | Refutação, sensibilidade |

In [ ]:
# Resumo consolidado do DAG v12
resumo = pd.DataFrame({
    'Elemento': [
        'Tratamento',
        'Outcome',
        'Confundidores (total)',
        'Confundidores → Tratamento',
        'Confundidores → Outcome',
        'Tratamento → Outcome',
        'DAG acíclico?',
        'Critério backdoor?',
        'Identificabilidade (DoWhy)',
    ],
    'Detalhe': [
        treatment,
        outcome,
        str(len(confounders)),
        str(len(confounders)),
        str(len(confounders)),
        '1',
        str(nx.is_directed_acyclic_graph(G_back)),
        'Satisfeito — nenhum confundidor é descendente do tratamento',
        'ATE via backdoor adjustment (confirmado pelo DoWhy)',
    ]
})
print('=' * 65)
print('RESUMO — DAG v12  |  is_delayed → review_score_outcome')
print('=' * 65)
print(resumo.to_string(index=False))